# 1. Archiving

**RadDB** turns radar volumes into a compact, queryable Parquet archive. A volume
is an [xarray](https://docs.xarray.dev/) `DataTree` (one group per sweep).

This notebook covers:

1. Raw data and RadDB initialisation
2. Archiving
3. Archived data

---
## How the archive is stored

A radar data is stored as **static data (LUT)** (per-gate geometry, computed once) plus
**dynamic data, one file per volume** (polarimetric variables), linked by an integer `gate_id`.
Following is an example of one archived volume (paths and files):

```
{archive_dir}/{radar}/LUT/{radar}_LUT.parquet          # gate centroids
{archive_dir}/{radar}/LUT/{radar}_h_plane_LUT.parquet  # horizontal gate plane (for PPI)
{archive_dir}/{radar}/LUT/{radar}_v_plane_LUT.parquet  # vertical gate plane  (for RHI)
{archive_dir}/{radar}/LUT/{radar}_corners_LUT.parquet  # 3-D gate corners
{archive_dir}/{radar}/LUT/{radar}_info.yaml            # site, CRS, scan geometry

{archive_dir}/{radar}/{YYYY}/{MM}/{DD}/{radar}_{YYYYMMDD}_{HHMMSS}_POL.parquet    # dynamic data
```

The geometry is stored **once**, not once per volume — which is what keeps the
archive small. Gates with no echo are dropped at archive time (`DBZH > 0` by
default).

In [ ]:
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path

import raddb
from raddb.lut import suggest_crs

print("raddb", raddb.__version__)

raddb 0.1.dev5+gde6070734.d20260323


## 1. Raw data and RadDB initialisation

### Input paths

`FMI_DIR` holds the DataTree volumes to be archived; `ARCHIVE_DIR` is where RadDB
writes the archive. Edit them to match your own machine.

In [ ]:
# --------------------------------------------------------------------------
# CONFIGURATION — point these at your own data
# --------------------------------------------------------------------------
# Any xarray DataTree with the standard xradar layout works.
# These tutorials use Finnish (FMI) volumes stored as zarr; FMI publishes them
# openly, so every example here can be reproduced.
# Edit the paths below to point at your own data.

FMI_DIR = Path("~/Desktop/LTE_project/ltenas8/data/RADAR/FMI_datatree_zarr").expanduser()
ARCHIVE_DIR = Path("~/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive").expanduser()

print("FMI DataTrees:", FMI_DIR)
print("Archive      :", ARCHIVE_DIR)

FMI DataTrees: /home/erik_poschivo/Desktop/LTE_project/ltenas8/data/RADAR/FMI_datatree_zarr
Archive      : /home/erik_poschivo/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive


### Inspecting raw data archive

`inventory(datatree_dir=...)` scans a directory of DataTree files and prints what
it finds: the radar name taken from each filename prefix, the number of files, the
time span they cover and their total size on disk.

In [ ]:
db = raddb.RadDB()
db.inventory(datatree_dir=FMI_DIR)

RadDB inventory — DataTree files on disk (not archived yet)
  directory : /home/erik_poschivo/Desktop/LTE_project/ltenas8/data/RADAR/FMI_datatree_zarr
  files     : 352
  radars    : FANJ, FKOR, FKUO  (from the filename prefix)
  time range: 2024-06-01 12:00:00 .. 2024-08-31 12:00:00
------------------------------------------------------------------------------
  radar     files  time range                                           size


  FANJ        122  2024-06-01 12:00:00 .. 2024-08-31 12:00:00         2.2 GB


  FKOR        115  2024-06-01 12:00:00 .. 2024-08-31 12:00:00         2.0 GB


  FKUO        115  2024-06-01 12:00:00 .. 2024-08-31 12:00:00         2.1 GB
------------------------------------------------------------------------------
  archive with: db.archive(datatree_dir='/home/erik_poschivo/Desktop/LTE_project/ltenas8/data/RADAR/FMI_datatree_zarr')


In [ ]:
# `detailed=True` adds a per-day breakdown
db.inventory(datatree_dir=FMI_DIR, detailed=True)

RadDB inventory — DataTree files on disk (not archived yet)
  directory : /home/erik_poschivo/Desktop/LTE_project/ltenas8/data/RADAR/FMI_datatree_zarr
  files     : 352
  radars    : FANJ, FKOR, FKUO  (from the filename prefix)
  time range: 2024-06-01 12:00:00 .. 2024-08-31 12:00:00
------------------------------------------------------------------------------
  radar     files  time range                                           size


  FANJ        122  2024-06-01 12:00:00 .. 2024-08-31 12:00:00         2.2 GB
      2024-06-01      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-02      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-03      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-04      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-05      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-06      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-07      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-08      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-09      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-10      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-12      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-13      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-14      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-15      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-16      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-17     24 volume(s)  12:00:00 .. 17:45:00
      2024-06-18      1 vol

  FKOR        115  2024-06-01 12:00:00 .. 2024-08-31 12:00:00         2.0 GB
      2024-06-01      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-02      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-03      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-04      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-05      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-06      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-07      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-08      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-09      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-10      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-11      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-12      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-13      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-14      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-15      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-16      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-17     24 vol

  FKUO        115  2024-06-01 12:00:00 .. 2024-08-31 12:00:00         2.1 GB
      2024-06-01      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-02      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-03      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-04      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-05      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-06      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-07      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-08      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-09      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-10      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-11      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-12      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-13      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-14      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-15      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-16      1 volume(s)  12:00:00 .. 12:00:00
      2024-06-17     24 vol

### Creating the RadDB object

`RadDB(archive_dir=..., crs=...)` returns an *archive-bound* RadDB: it knows where
the archive lives and which projection to write it in. This is the object used to
archive, open and inspect data.

In [ ]:
db = raddb.RadDB(archive_dir=ARCHIVE_DIR, crs=3067)  # 3067 ==> ETRS89 / TM35FIN (all of Finland)
db

RadDB(archive_dir=/home/erik_poschivo/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive, crs=3067) [archive-bound, no data loaded]

In [ ]:
# The same archive, opened without a CRS: reading never needs one.
raddb.RadDB(archive_dir=ARCHIVE_DIR)

RadDB(archive_dir=/home/erik_poschivo/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive, crs=None) [archive-bound, no data loaded]

## 2. Archiving

`archive()` takes either a directory of DataTree files or an in-memory DataTree.
The LUT is generated automatically from the first volume of each radar.

The window below is a convective afternoon over southern Finland — **17 June
2024**, sampled every quarter hour from 12:00 to 17:45 UTC on all three radars,
with the daily 12:00 volumes of the surrounding week for context. It is the case
the rest of these tutorials plot.

In [ ]:
result = db.archive(datatree_dir=FMI_DIR, time_period=("2024-06-10", "2024-06-18"))
result

RadDB archive
  archive_dir : /home/erik_poschivo/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive
  crs         : 3067
  radars      : ['FANJ', 'FKOR', 'FKUO']
  filter      : keep DBZH > 0.0
  volumes     : 92 archived, 0 failed
  elapsed     : 4m 50s


{'n_archived': 92,
 'n_failed': 0,
 'n_skipped': 0,
 'radars': ['FANJ', 'FKOR', 'FKUO']}

### The CRS constraint

**A projection is mandatory to write an archive, and never needed to read one. It can be given when RadDB is initialised or at archiving time.**

The LUT stores projected gate coordinates, and every crop and cross-section is
computed in them. A wrong projection is therefore silently wrong: the crop still
returns gates, the plot still looks like weather, and only the distances are
false. RadDB has no default — the CRS is stated once and is **measured** against
the radar's real position before anything is written: a 100 km geodesic is
projected in eight directions at the site and compared with the truth.

Two thresholds decide the outcome: above **0.1 %** RadDB warns, above **1 %** it
refuses and the whole archive is aborted.

In [ ]:
from raddb.lut import crs_distance_error

# The three FMI radars, and the projections one might reach for.
SITES = {"FANJ": (27.108, 60.904), "FKOR": (21.643, 60.128), "FKUO": (27.382, 62.863)}
CANDIDATES = {
    3067: "ETRS89 / TM35FIN — the Finnish national grid",
    32634: "WGS 84 / UTM zone 34N",
    32635: "WGS 84 / UTM zone 35N",
    2056: "CH1903+ / LV95 — Switzerland",
    3857: "WGS 84 / Pseudo-Mercator — the web-map projection",
}

print(f"{'EPSG':<7} {'FANJ':>8} {'FKOR':>8} {'FKUO':>8}   distortion of a 100 km baseline")
for epsg, label in CANDIDATES.items():
    errs = [crs_distance_error(epsg, longitude=lon, latitude=lat) for lon, lat in SITES.values()]
    verdict = "REFUSED" if max(errs) > 1.0 else ("warns" if max(errs) > 0.1 else "ok")
    print(f"{epsg:<7} " + " ".join(f"{e:7.3f}%" for e in errs) + f"   {verdict:<8} {label}")

EPSG        FANJ     FKOR     FKUO   distortion of a 100 km baseline


3067      0.040%   0.109%   0.040%   warns    ETRS89 / TM35FIN — the Finnish national grid
32634     0.139%   0.039%   0.133%   warns    WGS 84 / UTM zone 34N
32635     0.040%   0.109%   0.040%   warns    WGS 84 / UTM zone 35N
2056      3.828%   3.182%   4.871%   REFUSED  CH1903+ / LV95 — Switzerland
3857    108.387% 103.387% 122.375%   REFUSED  WGS 84 / Pseudo-Mercator — the web-map projection


In [ ]:
# `suggest_crs()` returns the UTM zone of a site — and Finland spans two of them.
# That is exactly why EPSG:3067 exists: one national grid, valid country-wide, so
# all three radars can be archived together and share one AOI frame.
for name, (lon, lat) in SITES.items():
    print(f"{name}: suggest_crs ==> EPSG:{suggest_crs(latitude=lat, longitude=lon)}")

# A projection from the wrong part of the world is refused, and the refusal aborts
# the whole archive — nothing is written.  (A throwaway directory, so the archive
# built above is untouched: a radar's LUT is validated when it is generated, and
# the one for FANJ already exists.)
import tempfile

with tempfile.TemporaryDirectory() as scratch:
    try:
        raddb.RadDB(archive_dir=scratch, crs=2056).archive(  # 2056 ==> CH1903+ / LV95
            datatree_dir=FMI_DIR,
            radar=["FANJ"],
            time_period=("2024-06-17", "2024-06-18"),
        )
    except ValueError as exc:
        print("\ncrs=2056 (Swiss LV95) on a Finnish radar:")
        print(" ", exc)
    print("\nPOL files written:", list(Path(scratch).rglob("*POL.parquet")))

FANJ: suggest_crs ==> EPSG:32635
FKOR: suggest_crs ==> EPSG:32634
FKUO: suggest_crs ==> EPSG:32635



crs=2056 (Swiss LV95) on a Finnish radar:
  EPSG:2056 (CH1903+ / LV95), valid for Liechtenstein; Switzerland. distorts distance by 3.8% at radar FANJ (27.1081, 60.9039) — gate geometry, crops and cross-sections would all be wrong by that much. Suggested for this site: EPSG:32635.

POL files written: []


## 3. Archived data

In [ ]:
db = raddb.RadDB(archive_dir=ARCHIVE_DIR)
print("radars in the archive:", db.list_radars())
db.inventory()

radars in the archive: ['FANJ', 'FKOR', 'FKUO']
RadDB inventory — archived data
  archive_dir : /home/erik_poschivo/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive
  radars      : FANJ, FKOR, FKUO
  volumes     : 92
  time range  : 2024-06-10 12:00:01 .. 2024-06-17 17:45:06
------------------------------------------------------------------------------
  radar   volumes  time range                                           size
  FANJ         30  2024-06-10 12:00:02 .. 2024-06-17 17:45:04       487.4 MB
  FKOR         31  2024-06-10 12:00:01 .. 2024-06-17 17:45:01       203.7 MB
  FKUO         31  2024-06-10 12:00:05 .. 2024-06-17 17:45:06       166.9 MB
------------------------------------------------------------------------------
  load with   : db.open(radars=..., time_period=(start, end))


In [ ]:
for p in sorted((ARCHIVE_DIR / "FANJ" / "LUT").iterdir()):
    print(f"  {p.name:<28} {p.stat().st_size / 1e6:8.2f} MB")

  FANJ_LUT.parquet                67.08 MB
  FANJ_corners_LUT.parquet        21.62 MB
  FANJ_h_plane_LUT.parquet        20.71 MB
  FANJ_info.yaml                   0.00 MB
  FANJ_v_plane_LUT.parquet         0.45 MB


### `gate_id`: how a volume finds its geometry

One int64 per gate links a row of data (polarimetric variables) to its row of geometry:

```
gate_id = radar_code * 10^12 + sweep * 10^10 + azimuth*10 * 10^6 + range_m
```

In [ ]:
lut = db.get_lut("FANJ")
print("\nLUT:", lut.shape)
print(lut.columns)
print(lut.head(5).select(["gate_id", "sweep", "azimuth", "range", "latitude", "longitude", "altitude"]))
print(lut.head(5).select(["x", "y", "z", "x_3067", "y_3067"]))


LUT: (1610280, 13)
['gate_id', 'sweep', 'azimuth', 'range', 'elevation_angle', 'latitude', 'longitude', 'altitude', 'x', 'y', 'z', 'x_3067', 'y_3067']
shape: (5, 7)
┌────────────────────┬───────┬─────────┬────────┬───────────┬───────────┬────────────┐
│ gate_id            ┆ sweep ┆ azimuth ┆ range  ┆ latitude  ┆ longitude ┆ altitude   │
│ ---                ┆ ---   ┆ ---     ┆ ---    ┆ ---       ┆ ---       ┆ ---        │
│ i64                ┆ i32   ┆ f64     ┆ f32    ┆ f64       ┆ f64       ┆ f64        │
╞════════════════════╪═══════╪═════════╪════════╪═══════════╪═══════════╪════════════╡
│ 713647000001000250 ┆ 0     ┆ 0.1     ┆ 250.0  ┆ 60.906118 ┆ 27.108068 ┆ 140.31267  │
│ 713647000001000750 ┆ 0     ┆ 0.1     ┆ 750.0  ┆ 60.910615 ┆ 27.108084 ┆ 142.960081 │
│ 713647000001001250 ┆ 0     ┆ 0.1     ┆ 1250.0 ┆ 60.915111 ┆ 27.1081   ┆ 145.636922 │
│ 713647000001001750 ┆ 0     ┆ 0.1     ┆ 1750.0 ┆ 60.919608 ┆ 27.108117 ┆ 148.343192 │
│ 713647000001002250 ┆ 0     ┆ 0.1     ┆ 2250.0 ┆ 6

### Radar site metadata

`info.yaml` records everything needed to reconstruct the geometry — including the
CRS that was validated at archive time, and the radar's **scan strategy**.

In [ ]:
info = db.get_radar_info("FANJ")
for k in ["radar", "network", "latitude", "longitude", "altitude", "crs", "ke", "beamwidth_deg", "n_sweeps", "n_gates"]:
    print(f"  {k:<16} {info[k]}")

  radar            FANJ
  network          
  latitude         60.90387001633644
  longitude        27.1080600656569
  altitude         139.0
  crs              {'epsg': 3067, 'columns': ['x_3067', 'y_3067']}
  ke               1.3333333333333333
  beamwidth_deg    1.0
  n_sweeps         13
  n_gates          1610280


---
**Next:** [2 — Opening and filtering](02_opening_and_filtering.ipynb)